In [1]:
import numpy as np
import networkx as nx
from scipy.ndimage import binary_dilation
import tifffile
import pandas as pd
import matplotlib.pyplot as plt
from shapely import Polygon
from skimage import measure
import networkx as nx
import skimage as ski
import dask.array as da
from shapely.strtree import STRtree
from scipy.spatial import ConvexHull, Delaunay, delaunay_plot_2d, convex_hull_plot_2d
import scipy.ndimage as ndi
import ast
from extract_features import subimage
from collections import defaultdict
from misc_utils import fixup_scipy_ndimage_result as fix
from misc_utils import strel_disk
import plotly.express as px
import dask_image.ndmeasure

In [ ]:
import importlib
import misc_utils
importlib.reload(misc_utils)
from misc_utils import strel_disk

In [ ]:
y0 = 4000
x0 = 10000
labels = tifffile.imread("/Users/hannahbolen/Desktop/image_analysis/masks_and_results/temp/reconcile_o8p_day24_s12_cyto_mask.tif")[y0:y0+4000, x0:x0+8400]

In [ ]:
def prepare_box_for_contours(box, shape, pad=3):
    """Marginally pads a bounding box so that object boundaries
    are not on cropped image patch edges.
    """
    for i in range(2):
        box[i] = max(0, box[i] - pad)
        box[i+2] = min(shape[i], box[i+2] + pad)
    slices = tuple([slice(box[i], box[i+2]) for i in range(2)])
    top_left = np.array(box[:2])[None] # (1, 2)
    return slices, top_left

def make_polygons_from_mask(mask):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    polygons = {}
    for rp in measure.regionprops(mask):
        # Faster to compute contours on small cell tiles than the whole image
        box_slices, box_top_left = prepare_box_for_contours(list(rp.bbox), mask.shape)
        label_mask = mask[box_slices] == rp.label

        label_cnts = np.concatenate(
            measure.find_contours(label_mask), axis=0
        )

        polygons[rp.label] = Polygon(label_cnts + box_top_left)
    
    return polygons

def make_polygons_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    polygons = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        polygons[lab] = Polygon(label_cnts + box_top_left)
    
    return polygons

def make_hull_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    hulls = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        hulls[lab] = ConvexHull(label_cnts + box_top_left)
    
    return hulls

def make_delaunay_from_boxes(mask, labels):
    """Constructs a polygon for each object in a mask. Returns
    a dict where each key is a label id and values are shapely polygons.
    """
    tris = {}
    for lab in labels:
        box = subimage(mask, cell_coords.at[lab, "cells_bounds"], pad=5)
        box_top_left = np.array(cell_coords.at[lab, "cells_bounds"][:2])[None]

        label_cnts = np.concatenate(
            measure.find_contours(box), axis=0
        )

        tris[lab] = Delaunay(label_cnts + box_top_left)
    
    return tris

def pairwise_polygon_distance(polygons_dict, dist):
    """Computes pairwise distance between all polygons in
    a dictionary. Returns a dictionary of distances.
    """
    polys = list(polygons_dict.values())
    ids = list(polygons_dict.keys())

    tree = STRtree(polys)

    id_map = dict(zip(polys, ids))

    distances = {i: {} for i in ids}

    for poly in polys:
        i = id_map[poly]

        # query nearby polygons using bounding boxes
        candidates = tree.query(poly.buffer(dist))  # 5 px search radius

        for other in candidates:
            j = id_map[other]
            if i == j:
                continue

            d = poly.distance(other)
            distances[i][j] = d
                
    return distances

def get_contour_from_label(mask, labels, df = cell_coords, scaled=True):
    contours = defaultdict(int)
    if not isinstance(labels, list):
        labels = list(labels)
    
    for label in labels:
        box = subimage(mask, df.at[label, "cells_bounds"], pad=5)

        label_cnts = np.concatenate(
        measure.find_contours(box), axis=0
            )
        
        if scaled:
            box_top_left = np.array(df.at[label, "cells_bounds"][:2])[None]
            contours[label] = label_cnts + box_top_left
        else:
            contours[label] = label_cnts
            
    return contours

In [ ]:
# labels,_,_ = ski.segmentation.relabel_sequential(labels, offset=1)
# labels = da.from_array(labels)
nobjects = da.max(labels).compute()
object_indexes = np.arange(nobjects, dtype=np.int32) + 1
distance = 25
strel = strel_disk(distance)

In [ ]:
d1, d2 = np.mgrid[0:10, 0:10]
d11, d22 = np.meshgrid(np.arange(10),np.arange(10))
np.array_equal(d1,d11)

d11 = np.transpose(d11,[1,0,2])
np.array_equal(d1,d11)

True

In [ ]:
i, j = np.mgrid[0 : labels.shape[0], 0 : labels.shape[1]]

minimums_i, maximums_i, _, _ = dask_image.ndmeasure.extrema(
    i, labels, object_indexes
)
minimums_j, maximums_j, _, _ = ndi.extrema(
    j, labels, object_indexes
)

minimums_i = np.maximum(fix(minimums_i) - distance, 0).astype(int)
maximums_i = np.minimum(
    fix(maximums_i) + distance + 1, labels.shape[0]
).astype(int)
minimums_j = np.maximum(fix(minimums_j) - distance, 0).astype(int)
maximums_j = np.minimum(
    fix(maximums_j) + distance + 1, labels.shape[1]
).astype(int)

In [ ]:
edges = set()

for object_number in object_indexes:
    if object_number == 0:
        #
        # No corresponding object in small-removed. This means
        # that the object has no pixels, e.g., not renumbered.
        #
        continue
    index = object_number - 1

    patch = labels[
        minimums_i[index] : maximums_i[index],
        minimums_j[index] : maximums_j[index],
    ]
    npatch = labels[
        minimums_i[index] : maximums_i[index],
        minimums_j[index] : maximums_j[index],
    ]

    #
    # Find the neighbors
    #
    patch_mask = patch == (index + 1)

    if distance <= 5:
        extended = ndi.binary_dilation(patch_mask, strel)
    else:
        extended = (
            scipy.signal.fftconvolve(patch_mask, strel, mode="same") > 0.5
        )

    neighbors = np.unique(npatch[extended])
    neighbors = neighbors[neighbors != 0]
    neighbors = neighbors[neighbors != object_number].compute()
    for n in neighbors:
        edges.add(tuple(sorted((n,object_number))))

In [ ]:
G = nx.Graph()
G.add_nodes_from(object_indexes)
G.add_edges_from(edges)
components = list(nx.connected_components(G))

In [ ]:
colony_map = {}
for k, comp in enumerate(components, start=1):
    if len(comp) < 3:
        for cell_id in comp:
            colony_map[cell_id] = -1
    else:
        for cell_id in comp:
            colony_map[cell_id] = k

In [ ]:
max_label = labels.max().compute()

label_to_colony = np.full(max_label + 1, 0, dtype=int)
for cell_id, colony_id in colony_map.items():
    label_to_colony[cell_id] = colony_id
    
colony_img = label_to_colony[labels]

ids = np.unique(colony_img)
ids = ids[ids>0]
cmap = plt.get_cmap("tab20", len(ids))

label_to_color = {
    lab: cmap(i)
    for i, lab in enumerate(ids)
}

# set background / non-colony
label_to_color[-1] = (0.7, 0.7, 0.7, 1)
label_to_color[0] = (0,0,0, 1)

colors_array = np.zeros((max_label + 1, 4))

for lab, color in label_to_color.items():
    colors_array[lab] = color

rgb = colors_array[colony_img]


plt.imshow(rgb)

In [ ]:
px.imshow(labels)

In [ ]:
threshold = 10
both_borders = ski.segmentation.find_boundaries(both, mode="inner")*both
dilated = ndi.grey_dilation(both_borders, size=(2*threshold+1, 2*threshold+1))
plt.imshow(dilated)

In [ ]:
threshold = 10
borders_region = ski.segmentation.find_boundaries(mask_region.compute())*mask_region
dilated_region = ndi.grey_dilation(borders_region, size=(2*threshold+1, 2*threshold+1))
plt.imshow(dilated_region)

In [ ]:
edges = set()

# horizontal neighbors
a = dilated_region[:, :-1]
b = dilated_region[:, 1:]

mask_diff = (a != b) & (a > 0) & (b > 0)

pairs = np.stack([a[mask_diff], b[mask_diff]], axis=1)

for i, j in pairs:
    if i != j:
        edges.add(tuple(sorted((i, j))))

a = dilated_region[:-1, :]
b = dilated_region[1:, :]

mask_diff = (a != b) & (a > 0) & (b > 0)

pairs = np.stack([a[mask_diff], b[mask_diff]], axis=1)

for i, j in pairs:
    if i != j:
        edges.add(tuple(sorted((i, j))))

G = nx.Graph()
G.add_nodes_from(np.unique(mask_region.compute())[1:])
G.add_edges_from(edges)
components = list(nx.connected_components(G))

colony_map = {}
for k, comp in enumerate(components, start=1):
    if len(comp) < 3:
        for cell_id in comp:
            colony_map[cell_id] = -1
    else:
        for cell_id in comp:
            colony_map[cell_id] = k

cell_coords["colony_id"] = cell_coords.index.map(colony_map)


In [ ]:
max_label = dilated_region.max()

label_to_colony = np.full(max_label + 1, 0, dtype=int)
for cell_id, colony_id in colony_map.items():
    label_to_colony[cell_id] = colony_id
    
colony_img = label_to_colony[dilated_region]

ids = np.unique(colony_img)
ids = ids[ids>0]
cmap = plt.get_cmap("tab20", len(ids))

label_to_color = {
    lab: cmap(i)
    for i, lab in enumerate(ids)
}

# set background / non-colony
label_to_color[-1] = (0.7, 0.7, 0.7, 1)
label_to_color[0] = (0,0,0, 1)

colors_array = np.zeros((max_label + 1, 4))

for lab, color in label_to_color.items():
    colors_array[lab] = color

rgb = colors_array[colony_img]


plt.imshow(rgb)

In [ ]:
colony_map = {}
for k, comp in enumerate(components, start=1):
    if len(comp) < 3:
        for cell_id in comp:
            colony_map[cell_id] = -1
    else:
        for cell_id in comp:
            colony_map[cell_id] = k

cell_coords["colony_id"] = cell_coords.index.map(colony_map)

# sizes = df["colony_id_poly"].value_counts()

# valid = sizes[sizes >= 10].index

# df["in_colony"] = df["colony_id_poly"].isin(valid)

In [ ]:
distances = pd.DataFrame(distances_dict)
distances_flat = distances.to_numpy().flatten()
distances_flat = distances_flat[~np.isnan(distances_flat)]

In [ ]:
G = nx.Graph()

# add all cells
G.add_nodes_from(polygons_dict.keys())

threshold = 20  # adjust

for i, neighbors in distances_dict.items():
    for j, d in neighbors.items():
        if d <= threshold:
            G.add_edge(i, j)

components = list(nx.connected_components(G))

colony_map = {}
for k, comp in enumerate(components):
    for cell_id in comp:
        colony_map[cell_id] = k

df["colony_id_poly"] = df.index.map(colony_map)

sizes = df["colony_id_poly"].value_counts()

valid = sizes[sizes >= 10].index

df["in_colony"] = df["colony_id_poly"].isin(valid)

In [ ]:
labels = df["colony_id_poly"].values
unique_labels = np.unique(labels)

# assign a color to each label
colors_map = {
    lab: plt.cm.Spectral(i / len(unique_labels))
    for i, lab in enumerate(unique_labels)
}

# override noise (-1) to black
filtered_dict = {key:(value if np.isin(key,valid)
          else (0, 0, 0, 1)) for key, value in colors_map.items()}

# build color list per point
colors = [filtered_dict[lab] for lab in labels]

fig, ax = plt.subplots(ncols=2,figsize=(20,10))

ax[0].scatter(
    df["j"],
    df["i"],
    c=colors,
    s=5
)

ax[0].set_xticks([])
ax[0].set_yticks([])
ax[0].invert_yaxis()

ax[1].imshow(ski.exposure.rescale_intensity(img[1], in_range=(0,3000)))
ax[1].set_xticks([])
ax[1].set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(ncols=2,figsize=(20,10))

for i in range(2):
    ax[i].imshow(ski.exposure.rescale_intensity(img[i], in_range=(0,3000)))
    ax[i].set_xticks([])
    ax[i].set_yticks([])

plt.tight_layout()
plt.show()